In [ ]:
# E4B fresh-retry pack (agent_e4b_R_reads): LoopAgent(3)[fixer, verifier], gate = existing tests (stale-bytecode fix), 80/30/200.
# Base measured: e4b-repro-ab ARM_F 11.50/39 vs plain ca5 9.82 (Welch t=3.18). CPU only.
import base64, hashlib, json, shutil, zipfile
from pathlib import Path
FILES = json.loads("""{"agent.yaml": "bmFtZTogYWdlbnRfZTRiX3JldHJ5X2ZyZXNoCmFnZW50X2NsYXNzOiBMb29wQWdlbnQKbWF4X2l0ZXJhdGlvbnM6IDMKZGVzY3JpcHRpb246IGZpeGVyIGF0dGVtcHRzLCBnYXRlIGNoZWNrcyBleGlzdGluZyB0ZXN0czsgb24gcmVqZWN0IHRoZSBhdHRlbXB0IGlzIHVuZG9uZSBhbmQgdGhlIGZpeGVyIHJldHJpZXMgZnJlc2ggKG1heCAzKTsgb25seSB0aGUgdmVyaWZpZXIgc3VibWl0cwpzdWJfYWdlbnRzOgogIC0gY29uZmlnX3BhdGg6IHN1Yl9hZ2VudHMvZml4ZXIueWFtbAogIC0gY29uZmlnX3BhdGg6IHN1Yl9hZ2VudHMvdmVyaWZpZXIueWFtbAo=", "eval_config.yaml": "ZXZhbHVhdGlvbjoKICBtYXhfdGltZV9taW51dGVzOiAzMAogIG1heF90b29sX2NhbGxzOiA4MAogIG1heF90dXJuczogMjAwCiAgdGltZW91dF9zZWNvbmRzOiAzMDAK", "skills/submit-gate/scripts/check.py": "IiIic3VibWl0LWdhdGU6IGRldGVybWluaXN0aWMgcHJlLXN1Ym1pdCBjaGVjaywgcnVuIGluc2lkZSB0aGUgdGFzayBzYW5kYm94IChjd2QgL3dvcmtzcGFjZSkuClByaW50cyBBQ0NFUFRFRCBvciBSRUpFQ1RFRCAoZmlyc3QgbGluZSkgcGx1cyBhIHNob3J0IHJlYXNvbi4gV3JpdGVzIG5vdGhpbmcgaW5zaWRlIC93b3Jrc3BhY2UgZXhjZXB0CnRyYW5zaWVudGx5IHJldmVydGluZy9yZS1hcHBseWluZyB0aGUgYWdlbnQncyBvd24gZGlmZiB0byBjb21wYXJlIGFnYWluc3QgdGhlIG9yaWdpbmFsIGNvZGUuCiIiIgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKCmRlZiBfZmluZF93b3Jrc3BhY2UoKToKICAgICMgVGhlIGhhcm5lc3Mgd3JpdGVzIHRoaXMgc2NyaXB0IGludG8gdGhlIHNhbmRib3ggd29ya19kaXIgKD0gdGhlIHJlcG8gcm9vdCkgYW5kIHJ1bnMgaXQgZnJvbSB0aGVyZS4KICAgICMgRG9ja2VyIHNhbmRib3g6IC93b3Jrc3BhY2UuIFN1YnByb2Nlc3Mgc2FuZGJveCAoS2FnZ2xlIEU0QiBrZXJuZWwsIGxvY2FsIHJpZyk6IGEgdGVtcCBkaXIuIFRyeSBpbiBvcmRlci4KICAgICMgQURLIHJ1bnMgc2tpbGwgc2NyaXB0cyBmcm9tIGEgdGVtcCBDT1BZIG9mIHRoZSBza2lsbCAoPHNhbmRib3g+L3RtcC90bXBYWFhYL3NjcmlwdHMvKSwgbm90IGZyb20gdGhlIHJlcG8uCiAgICAjIFN1YnByb2Nlc3Mgc2FuZGJveCBleHBvcnRzIFBZVEhPTlBBVEg9PHNhbmRib3g+L3dvcmtzcGFjZSBhbmQgSE9NRT08c2FuZGJveD4vdG1wOyBkb2NrZXIgc2FuZGJveCB1c2VzIC93b3Jrc3BhY2UuCiAgICBjYW5kcyA9IFtwIGZvciBwIGluIG9zLmVudmlyb24uZ2V0KCJQWVRIT05QQVRIIiwgIiIpLnNwbGl0KG9zLnBhdGhzZXApIGlmIHBdCiAgICBjYW5kcyArPSBbIi93b3Jrc3BhY2UiXQogICAgaG9tZSA9IG9zLmVudmlyb24uZ2V0KCJIT01FIiwgIiIpCiAgICBpZiBob21lOgogICAgICAgIGNhbmRzLmFwcGVuZChvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKGhvbWUucnN0cmlwKCIvIikpLCAid29ya3NwYWNlIikpCiAgICBkID0gb3MucGF0aC5hYnNwYXRoKG9zLmdldGN3ZCgpKQogICAgd2hpbGUgZCBhbmQgZCAhPSAiLyI6CiAgICAgICAgY2FuZHMuYXBwZW5kKG9zLnBhdGguam9pbihkLCAid29ya3NwYWNlIikpCiAgICAgICAgZCA9IG9zLnBhdGguZGlybmFtZShkKQogICAgY2FuZHMgKz0gW29zLmdldGN3ZCgpXQogICAgZm9yIGMgaW4gY2FuZHM6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByID0gc3VicHJvY2Vzcy5ydW4oImdpdCByZXYtcGFyc2UgLS1zaG93LXRvcGxldmVsIiwgc2hlbGw9VHJ1ZSwgY3dkPWMsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD0xMCkKICAgICAgICAgICAgaWYgci5yZXR1cm5jb2RlID09IDAgYW5kIHIuc3Rkb3V0LnN0cmlwKCk6CiAgICAgICAgICAgICAgICByZXR1cm4gci5zdGRvdXQuc3RyaXAoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgIHJldHVybiBjYW5kc1swXQoKClcgPSBfZmluZF93b3Jrc3BhY2UoKQpUX1BZVEVTVCA9IDkwCgoKX0VOViA9IGRpY3Qob3MuZW52aXJvbiwgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9IjEiKQoKCmRlZiBfcHVyZ2VfcHljKCk6CiAgICBzdWJwcm9jZXNzLnJ1bigiZmluZCAuIC1wYXRoIC4vLmdpdCAtcHJ1bmUgLW8gLW5hbWUgX19weWNhY2hlX18gLXR5cGUgZCAtcHJ1bmUgLWV4ZWMgcm0gLXJmIHt9ICsgMj4vZGV2L251bGwiLAogICAgICAgICAgICAgICAgICAgc2hlbGw9VHJ1ZSwgY3dkPVcsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRpbWVvdXQ9NjApCgoKZGVmIHNoKGNtZCwgdGltZW91dD02MCk6CiAgICBpZiAicHl0ZXN0IiBpbiBjbWQgb3IgInJlcHJvX2lzc3VlIiBpbiBjbWQ6CiAgICAgICAgX3B1cmdlX3B5YygpCiAgICB0cnk6CiAgICAgICAgciA9IHN1YnByb2Nlc3MucnVuKGNtZCwgc2hlbGw9VHJ1ZSwgY3dkPVcsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD10aW1lb3V0LCBlbnY9X0VOVikKICAgICAgICByZXR1cm4gci5yZXR1cm5jb2RlLCAoci5zdGRvdXQgb3IgIiIpICsgKHIuc3RkZXJyIG9yICIiKQogICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgcmV0dXJuIDEyNCwgIlRJTUVPVVQiCgoKZGVmIGRpZmZfYmFzZSgpOgogICAgc2goImdpdCBhZGQgLU4gLiA+L2Rldi9udWxsIDI+JjEiKQogICAgcmMsIF8gPSBzaCgiZ2l0IHJldi1wYXJzZSAtcSAtLXZlcmlmeSBfc3dlZ2VtbWFfYmFzZWxpbmUgPi9kZXYvbnVsbCAyPiYxIikKICAgIHJldHVybiAiX3N3ZWdlbW1hX2Jhc2VsaW5lIiBpZiByYyA9PSAwIGVsc2UgIkhFQUQiCgoKZGVmIGlzX3Rlc3QocCk6CiAgICBiID0gb3MucGF0aC5iYXNlbmFtZShwKQogICAgcmV0dXJuIChiLnN0YXJ0c3dpdGgoInRlc3RfIikgb3IgYi5lbmRzd2l0aCgiX3Rlc3QucHkiKSBvciBiID09ICJjb25mdGVzdC5weSIKICAgICAgICAgICAgb3IgIi90ZXN0cy8iIGluICIvIiArIHAgb3IgIi90ZXN0LyIgaW4gIi8iICsgcCkKCgpkZWYgcHl0ZXN0X2ZhaWxfaWRzKGZpbGVzKToKICAgIHJjLCBvdXQgPSBzaCgidGltZW91dCAlZCBweXRob24zIC1tIHB5dGVzdCAtcSAtLW5vLWhlYWRlciAtcCBubzpjYWNoZXByb3ZpZGVyIC1vIGFkZG9wdHM9JycgLXJmRSAtLW1heGZhaWw9MjUgJXMiCiAgICAgICAgICAgICAgICAgJSAoVF9QWVRFU1QsICIgIi5qb2luKGZpbGVzKSksIHRpbWVvdXQ9VF9QWVRFU1QgKyAxNSkKICAgIGlkcyA9IHNldChyZS5maW5kYWxsKHIiXig/OkZBSUxFRHxFUlJPUikgKFxTKykiLCBvdXQsIHJlLk0pKQogICAgY29sbGVjdF9lcnIgPSAoImVycm9yIiBpbiBvdXQubG93ZXIoKSBhbmQgImNvbGxlY3RlZCAwIGl0ZW1zIiBpbiBvdXQpIG9yICJFUlJPUiBjb2xsZWN0aW5nIiBpbiBvdXQgb3IgIkltcG9ydEVycm9yIiBpbiBvdXRbLTIwMDA6XQogICAgcmV0dXJuIHJjLCBpZHMsIGNvbGxlY3RfZXJyLCBvdXQKCgpkZWYgbWFpbigpOgogICAgYmFzZSA9IGRpZmZfYmFzZSgpCiAgICByYywgbmFtZXMgPSBzaCgiZ2l0IGRpZmYgLS1uYW1lLW9ubHkgJXMiICUgYmFzZSkKICAgIGhhcm5lc3NfZmlsZXMgPSB7ImNvbmZ0ZXN0LnB5IiwgInB5dGVzdC5pbmkiLCAiX3N3ZWdlbW1hX3N0dWJzLnB5IiwgInNpdGVjdXN0b21pemUucHkiLCAidXNlcmN1c3RvbWl6ZS5weSJ9CiAgICBjaGFuZ2VkID0gW24uc3RyaXAoKSBmb3IgbiBpbiBuYW1lcy5zcGxpdGxpbmVzKCkgaWYgbi5zdHJpcCgpCiAgICAgICAgICAgICAgIGFuZCBub3Qgb3MucGF0aC5iYXNlbmFtZShuKS5zdGFydHN3aXRoKCIuYWRrX2V4ZWNfIikgYW5kIG4uc3RyaXAoKSBub3QgaW4gaGFybmVzc19maWxlc10KICAgIHNyYyA9IFtuIGZvciBuIGluIGNoYW5nZWQgaWYgbi5lbmRzd2l0aCgiLnB5IikgYW5kIG5vdCBpc190ZXN0KG4pXQogICAgaWYgcmMgIT0gMDoKICAgICAgICBfLCBncyA9IHNoKCJnaXQgc3RhdHVzIDI+JjEgfCBoZWFkIC0zOyBscyAtYSB8IGhlYWQgLTEyOyBlbnYgfCBncmVwIC1pRSAnXihHSVR8SE9NRXxQV0R8VVNFUiknIHwgaGVhZCIpCiAgICAgICAgcHJpbnQoIkFDQ0VQVEVEOiBHQVRFX0VSUk9SIGdpdCB1bmF2YWlsYWJsZSAocmM9JXMpIFc9JXMgY3dkPSVzIGZpbGU9JXMgfCAlcyB8ICVzIC0gY2hlY2sgc2tpcHBlZDsgc3VibWl0IGFzIGlzLiIKICAgICAgICAgICAgICAlIChyYywgVywgb3MuZ2V0Y3dkKCksIG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXyksIG5hbWVzLnN0cmlwKClbOjIwMF0ucmVwbGFjZSgiXG4iLCAiICIpLCBncy5yZXBsYWNlKCJcbiIsICIgIilbOjYwMF0pKQogICAgICAgIHJldHVybgogICAgaWYgbm90IGNoYW5nZWQ6CiAgICAgICAgcHJpbnQoIlJFSkVDVEVEOiBubyBjaGFuZ2UgaW4gdGhlIHJlcG8gLSB5b3VyIGVkaXQgZGlkIG5vdCBsYW5kLiBSZS1hcHBseSBpdCB3aXRoIGVkaXRfZmlsZS4iKQogICAgICAgIHJldHVybgogICAgaWYgbm90IHNyYzoKICAgICAgICBwcmludCgiUkVKRUNURUQ6IG9ubHkgbm9uLXNvdXJjZS90ZXN0IGZpbGVzIGNoYW5nZWQgKCVzKS4gRWRpdCB0aGUgU09VUkNFIGZpbGUgaW5zdGVhZDsgdGVzdCBlZGl0cyBhcmUgcmVzZXQgYnkgdGhlIGdyYWRlci4iICUgIiwgIi5qb2luKGNoYW5nZWRbOjVdKSkKICAgICAgICByZXR1cm4KICAgICMgMS4gc3ludGF4CiAgICBmb3IgZiBpbiBzcmM6CiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihXLCBmKSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmMsIG91dCA9IHNoKCJweXRob24zIC1jIFwiaW1wb3J0IGFzdCxzeXM7IGFzdC5wYXJzZShvcGVuKHN5cy5hcmd2WzFdKS5yZWFkKCksIHN5cy5hcmd2WzFdKVwiICVzIiAlIGYpCiAgICAgICAgaWYgcmMgIT0gMDoKICAgICAgICAgICAgcHJpbnQoIlJFSkVDVEVEOiBzeW50YXggZXJyb3IgaW4gJXNcbiVzIiAlIChmLCAiXG4iLmpvaW4ob3V0LnN0cmlwKCkuc3BsaXRsaW5lcygpWy02Ol0pKSkKICAgICAgICAgICAgcmV0dXJuCiAgICAjIDIuIGV4aXN0aW5nIHRlc3RzIHRoYXQgZXhlcmNpc2UgdGhlIGVkaXRlZCBtb2R1bGUocykKICAgIHRkaXJzID0gW2QgZm9yIGQgaW4gKCJ0ZXN0cyIsICJ0ZXN0IikgaWYgb3MucGF0aC5pc2Rpcihvcy5wYXRoLmpvaW4oVywgZCkpXQogICAgaWYgbm90IHRkaXJzOgogICAgICAgIHByaW50KCJBQ0NFUFRFRDogc3ludGF4IE9LOyBubyB0ZXN0cy8gZGlyZWN0b3J5IHRvIGNoZWNrIGFnYWluc3QuIikKICAgICAgICByZXR1cm4KICAgIHBpY2tlZCA9IFtdCiAgICBmb3IgZiBpbiBzcmM6CiAgICAgICAgc3RlbSA9IG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZShmKSlbMF0KICAgICAgICBrZXkgPSBzdGVtLmxzdHJpcCgiXyIpCiAgICAgICAgZG90dGVkID0gb3MucGF0aC5zcGxpdGV4dChmKVswXS5yZXBsYWNlKCIvIiwgIi4iKQogICAgICAgIHJjLCBvdXQgPSBzaCgibHMgJXMgMj4vZGV2L251bGwgfCBncmVwIC1FICd0ZXN0Xz8lcyhcXC5weXxfKScgfCBoZWFkIC0yIiAlICgiICIuam9pbihkICsgIi8qIiBmb3IgZCBpbiB0ZGlycyksIHJlLmVzY2FwZShrZXkpKSkKICAgICAgICBjYW5kcyA9IFtjIGZvciBjIGluIG91dC5zcGxpdCgpIGlmIGMuZW5kc3dpdGgoIi5weSIpXQogICAgICAgIGlmIG5vdCBjYW5kczoKICAgICAgICAgICAgcmMsIG91dCA9IHNoKCJncmVwIC1ybEUgJyhmcm9tfGltcG9ydCkgKyVzXFxifGZyb20gKyVzICtpbXBvcnQgKyVzXFxiJyAlcyAtLWluY2x1ZGU9J3Rlc3RfKi5weScgMj4vZGV2L251bGwgfCBoZWFkIC0yIgogICAgICAgICAgICAgICAgICAgICAgICAgJSAocmUuZXNjYXBlKGRvdHRlZCksIHJlLmVzY2FwZSgiLiIuam9pbihkb3R0ZWQuc3BsaXQoIi4iKVs6LTFdKSBvciBkb3R0ZWQpLCByZS5lc2NhcGUoc3RlbSksICIgIi5qb2luKHRkaXJzKSkpCiAgICAgICAgICAgIGNhbmRzID0gW2MgZm9yIGMgaW4gb3V0LnNwbGl0KCkgaWYgYy5lbmRzd2l0aCgiLnB5IildCiAgICAgICAgZm9yIGMgaW4gY2FuZHM6CiAgICAgICAgICAgIGlmIGMgbm90IGluIHBpY2tlZDoKICAgICAgICAgICAgICAgIHBpY2tlZC5hcHBlbmQoYykKICAgIHBpY2tlZCA9IHBpY2tlZFs6Ml0KICAgIGlmIG5vdCBwaWNrZWQ6CiAgICAgICAgcHJpbnQoIkFDQ0VQVEVEOiBzeW50YXggT0s7IG5vIGV4aXN0aW5nIHRlc3QgZmlsZSBjb3ZlcnMgJXMuIiAlICIsICIuam9pbihzcmMpKQogICAgICAgIHJldHVybgogICAgcmNfcCwgZmFpbF9wLCBjZXJyX3AsIG91dF9wID0gcHl0ZXN0X2ZhaWxfaWRzKHBpY2tlZCkKICAgIGlmIHJjX3AgPT0gMDoKICAgICAgICBwcmludCgiQUNDRVBURUQ6IHN5bnRheCBPSzsgZXhpc3RpbmcgdGVzdHMgcGFzcyAoJXMpLiIgJSAiLCAiLmpvaW4ocGlja2VkKSkKICAgICAgICByZXR1cm4KICAgICMgMy4gY29tcGFyZSBhZ2FpbnN0IHRoZSBvcmlnaW5hbCBjb2RlIHNvIHByZS1leGlzdGluZyBmYWlsdXJlcyBhcmUgbm90IGJsYW1lZCBvbiB0aGUgZWRpdAogICAgIyBSZXZlcnQgT05MWSB0aGUgZWRpdGVkIHNvdXJjZSBmaWxlcyAobmV2ZXIgdGhlIGhhcm5lc3MncyBjb25mdGVzdC5weSAvIHB5dGVzdC5pbmkpLCB2aWEgYSBwZXItcnVuIHRlbXAgZmlsZQogICAgIyAocGxhaW4gL3RtcCBpcyBzaGFyZWQgYnkgY29uY3VycmVudCBzdWJwcm9jZXNzIHNhbmRib3hlcykuCiAgICBpbXBvcnQgdGVtcGZpbGUKICAgIGZkLCBnZGlmZiA9IHRlbXBmaWxlLm1rc3RlbXAocHJlZml4PSJfZ2F0ZV8iLCBzdWZmaXg9Ii5kaWZmIikKICAgIG9zLmNsb3NlKGZkKQogICAgcmMsIGQgPSBzaCgiZ2l0IGRpZmYgLS1iaW5hcnkgJXMgLS0gJXMgPiAlcyAmJiBlY2hvIG9rIiAlIChiYXNlLCAiICIuam9pbihzcmMpLCBnZGlmZikpCiAgICBiYXNlX2ZhaWwsIGNlcnJfYiA9IHNldCgpLCBGYWxzZQogICAgaWYgIm9rIiBpbiBkIGFuZCBvcy5wYXRoLmdldHNpemUoZ2RpZmYpID4gMDoKICAgICAgICBzaCgiZ2l0IGFwcGx5IC1SICVzIiAlIGdkaWZmKQogICAgICAgIHRyeToKICAgICAgICAgICAgXywgYmFzZV9mYWlsLCBjZXJyX2IsIF8gPSBweXRlc3RfZmFpbF9pZHMocGlja2VkKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHJjMiwgbzIgPSBzaCgiZ2l0IGFwcGx5ICVzIiAlIGdkaWZmKQogICAgICAgICAgICBpZiByYzIgIT0gMDoKICAgICAgICAgICAgICAgIHNoKCJnaXQgYXBwbHkgLS0zd2F5ICVzIiAlIGdkaWZmKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBvcy51bmxpbmsoZ2RpZmYpCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgcGFzcwogICAgaWYgY2Vycl9wIGFuZCBjZXJyX2I6CiAgICAgICAgcHJpbnQoIkFDQ0VQVEVEOiBzeW50YXggT0s7IGV4aXN0aW5nIHRlc3RzIGNhbm5vdCBiZSBjb2xsZWN0ZWQgaW4gdGhpcyBzYW5kYm94IG9uIHRoZSBvcmlnaW5hbCBjb2RlIGVpdGhlciAoJXMpLiIgJSAiLCAiLmpvaW4ocGlja2VkKSkKICAgICAgICByZXR1cm4KICAgIG5ldyA9IHNvcnRlZChmYWlsX3AgLSBiYXNlX2ZhaWwpCiAgICBpZiBjZXJyX3AgYW5kIG5vdCBjZXJyX2I6CiAgICAgICAgdGFpbCA9ICJcbiIuam9pbihvdXRfcC5zdHJpcCgpLnNwbGl0bGluZXMoKVstMTQ6XSkKICAgICAgICBwcmludCgiUkVKRUNURUQ6IHlvdXIgZWRpdCBicmVha3MgaW1wb3J0L2NvbGxlY3Rpb24gb2YgJXMgKGl0IGNvbGxlY3RlZCBvbiB0aGUgb3JpZ2luYWwgY29kZSkuXG4lcyIgJSAoIiwgIi5qb2luKHBpY2tlZCksIHRhaWwpKQogICAgICAgIHJldHVybgogICAgaWYgbm90IG5ldzoKICAgICAgICBwcmludCgiQUNDRVBURUQ6IHN5bnRheCBPSzsgbm8gTkVXIHRlc3QgZmFpbHVyZXMgdnMgdGhlIG9yaWdpbmFsIGNvZGUgKCVzKS4iICUgIiwgIi5qb2luKHBpY2tlZCkpCiAgICAgICAgcmV0dXJuCiAgICBmaXJzdCA9IG5ld1swXQogICAgbSA9IHJlLnNlYXJjaChyIl97NSx9IC4qPyVzLio/IF97NSx9XG4oLio/KSg/PVxuX3s1LH0gfFxuPSsgc2hvcnQgdGVzdCBzdW1tYXJ5KSIgJSByZS5lc2NhcGUoZmlyc3Quc3BsaXQoIjo6IilbLTFdKSwgb3V0X3AsIHJlLlMpCiAgICBkZXRhaWwgPSAobS5ncm91cCgxKSBpZiBtIGVsc2Ugb3V0X3ApWy0xNDAwOl0KICAgIHByaW50KCJSRUpFQ1RFRDogeW91ciBlZGl0IG1ha2VzICVkIGV4aXN0aW5nIHRlc3QocykgZmFpbCB0aGF0IHBhc3NlZCBvbiB0aGUgb3JpZ2luYWwgY29kZTogJXNcbi0tLSBmaXJzdCBmYWlsdXJlIC0tLVxuJXMiCiAgICAgICAgICAlIChsZW4obmV3KSwgIiwgIi5qb2luKG5ld1s6NF0pLCBkZXRhaWwpKQoKCk1BWF9BVFRFTVBUUyA9IDMKClJFUFJPID0gRmFsc2UKUkVQUk9fTkFNRSA9ICJyZXByb19pc3N1ZS5weSIKCgpkZWYgX3N0YXRlX2RpcigpOgogICAgaW1wb3J0IHRlbXBmaWxlCiAgICBkID0gb3MucGF0aC5qb2luKG9zLmVudmlyb24uZ2V0KCJIT01FIikgb3IgdGVtcGZpbGUuZ2V0dGVtcGRpcigpLCAiLnN1Ym1pdF9nYXRlIikKICAgIG9zLm1ha2VkaXJzKGQsIGV4aXN0X29rPVRydWUpCiAgICByZXR1cm4gZAoKCmRlZiBfcmVsb2NhdGVfcmVwcm8oKToKICAgICIiIk1vdmUgdGhlIHRlc3RlcidzIHNjcmlwdCBvdXQgb2YgdGhlIHdvcmtzcGFjZSBzbyBpdCBuZXZlciBsYW5kcyBpbiB0aGUgcGF0Y2guIiIiCiAgICBpbXBvcnQgc2h1dGlsCiAgICBzcmMgPSBvcy5wYXRoLmpvaW4oVywgUkVQUk9fTkFNRSkKICAgIGRzdCA9IG9zLnBhdGguam9pbihfc3RhdGVfZGlyKCksIFJFUFJPX05BTUUpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhzcmMpOgogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhkc3QpOgogICAgICAgICAgICBzaHV0aWwubW92ZShzcmMsIGRzdCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBvcy51bmxpbmsoc3JjKQogICAgcmV0dXJuIGRzdCBpZiBvcy5wYXRoLmV4aXN0cyhkc3QpIGVsc2UgTm9uZQoKCmRlZiBfcnVuX3JlcHJvKHBhdGgpOgogICAgcmMsIG91dCA9IHNoKCJ0aW1lb3V0IDYwIHB5dGhvbjMgJXMiICUgcGF0aCwgdGltZW91dD03NSkKICAgIHJldHVybiBvdXQKCgpkZWYgX3JlcHJvX3ZlcmRpY3QoKToKICAgICIiIlJldHVybnMgKHZhbGlkLCByZXNvbHZlZCwgdGFpbCkgb3IgTm9uZSB3aGVuIG5vIHJlcHJvIHNjcmlwdCBleGlzdHMuIiIiCiAgICBpbXBvcnQganNvbgogICAgaW1wb3J0IHRlbXBmaWxlCiAgICBwYXRoID0gX3JlbG9jYXRlX3JlcHJvKCkKICAgIGlmIG5vdCBwYXRoOgogICAgICAgIHJldHVybiBOb25lCiAgICBzcCA9IG9zLnBhdGguam9pbihfc3RhdGVfZGlyKCksICJyZXByby5qc29uIikKICAgIHN0ID0ganNvbi5sb2FkKG9wZW4oc3ApKSBpZiBvcy5wYXRoLmV4aXN0cyhzcCkgZWxzZSB7fQogICAgYmFzZSA9IGRpZmZfYmFzZSgpCiAgICBzcmMgPSBfc3JjX2ZpbGVzKGJhc2UpCiAgICBpZiAiYmFzZV9vdXQiIG5vdCBpbiBzdDoKICAgICAgICBmZCwgZyA9IHRlbXBmaWxlLm1rc3RlbXAocHJlZml4PSJfcmVwcm9fIiwgc3VmZml4PSIuZGlmZiIpCiAgICAgICAgb3MuY2xvc2UoZmQpCiAgICAgICAgcmV2ZXJ0ZWQgPSBGYWxzZQogICAgICAgIGlmIHNyYzoKICAgICAgICAgICAgc2goImdpdCBkaWZmIC0tYmluYXJ5ICVzIC0tICVzID4gJXMiICUgKGJhc2UsICIgIi5qb2luKHNyYyksIGcpKQogICAgICAgICAgICBpZiBvcy5wYXRoLmdldHNpemUoZykgPiAwOgogICAgICAgICAgICAgICAgc2goImdpdCBhcHBseSAtUiAlcyIgJSBnKQogICAgICAgICAgICAgICAgcmV2ZXJ0ZWQgPSBUcnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBzdFsiYmFzZV9vdXQiXSA9IF9ydW5fcmVwcm8ocGF0aClbLTE1MDA6XQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHJldmVydGVkOgogICAgICAgICAgICAgICAgcmMyLCBfID0gc2goImdpdCBhcHBseSAlcyIgJSBnKQogICAgICAgICAgICAgICAgaWYgcmMyICE9IDA6CiAgICAgICAgICAgICAgICAgICAgc2goImdpdCBhcHBseSAtLTN3YXkgJXMiICUgZykKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgb3MudW5saW5rKGcpCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgcGFzcwogICAgICAgIGpzb24uZHVtcChzdCwgb3BlbihzcCwgInciKSkKICAgIGIgPSBzdFsiYmFzZV9vdXQiXQogICAgdmFsaWQgPSAoIklzc3VlIHJlcHJvZHVjZWQiIGluIGIpIGFuZCAoIklzc3VlIHJlc29sdmVkIiBub3QgaW4gYikgYW5kICgiT3RoZXIgaXNzdWVzIiBub3QgaW4gYikKICAgIGlmIG5vdCB2YWxpZDoKICAgICAgICByZXR1cm4gKEZhbHNlLCBGYWxzZSwgYlstMzAwOl0pCiAgICBwID0gX3J1bl9yZXBybyhwYXRoKQogICAgcmV0dXJuIChUcnVlLCAiSXNzdWUgcmVzb2x2ZWQiIGluIHAsIHBbLTcwMDpdKQoKCmRlZiBfc3JjX2ZpbGVzKGJhc2UpOgogICAgaGFybmVzc19maWxlcyA9IHsiY29uZnRlc3QucHkiLCAicHl0ZXN0LmluaSIsICJfc3dlZ2VtbWFfc3R1YnMucHkiLCAic2l0ZWN1c3RvbWl6ZS5weSIsICJ1c2VyY3VzdG9taXplLnB5In0KICAgIF8sIG5hbWVzID0gc2goImdpdCBkaWZmIC0tbmFtZS1vbmx5ICVzIiAlIGJhc2UpCiAgICByZXR1cm4gW24uc3RyaXAoKSBmb3IgbiBpbiBuYW1lcy5zcGxpdGxpbmVzKCkgaWYgbi5zdHJpcCgpLmVuZHN3aXRoKCIucHkiKSBhbmQgbm90IGlzX3Rlc3Qobi5zdHJpcCgpKQogICAgICAgICAgICBhbmQgbi5zdHJpcCgpIG5vdCBpbiBoYXJuZXNzX2ZpbGVzIGFuZCBub3Qgb3MucGF0aC5iYXNlbmFtZShuKS5zdGFydHN3aXRoKCIuYWRrX2V4ZWNfIildCgoKZGVmIF9zY29yZShtc2cpOgogICAgIyBsb3dlciBpcyBiZXR0ZXI6IGhvdyBiYWQgd2FzIHRoaXMgYXR0ZW1wdAogICAgaWYgIlJFUFJPX0ZBSUwiIGluIG1zZzoKICAgICAgICBtID0gcmUuc2VhcmNoKHIibWFrZXMgKFxkKykgZXhpc3RpbmcgdGVzdCIsIG1zZykKICAgICAgICByZXR1cm4gNTAgKyAoaW50KG0uZ3JvdXAoMSkpIGlmIG0gZWxzZSAwKQogICAgaWYgbXNnLnN0YXJ0c3dpdGgoIkFDQ0VQVEVEIik6CiAgICAgICAgcmV0dXJuIDAKICAgIG0gPSByZS5zZWFyY2gociJtYWtlcyAoXGQrKSBleGlzdGluZyB0ZXN0IiwgbXNnKQogICAgaWYgbToKICAgICAgICByZXR1cm4gaW50KG0uZ3JvdXAoMSkpCiAgICBpZiAiYnJlYWtzIGltcG9ydC9jb2xsZWN0aW9uIiBpbiBtc2c6CiAgICAgICAgcmV0dXJuIDUwMAogICAgaWYgInN5bnRheCBlcnJvciIgaW4gbXNnOgogICAgICAgIHJldHVybiA5MDAKICAgIHJldHVybiAxMDAwICAjIG5vIHNvdXJjZSBjaGFuZ2UKCgpkZWYgX2ZpbmFsaXplKG1zZyk6CiAgICBpbXBvcnQganNvbgogICAgaW1wb3J0IHRlbXBmaWxlCiAgICBzdGF0ZV9kaXIgPSBvcy5wYXRoLmpvaW4ob3MuZW52aXJvbi5nZXQoIkhPTUUiKSBvciB0ZW1wZmlsZS5nZXR0ZW1wZGlyKCksICIuc3VibWl0X2dhdGUiKQogICAgb3MubWFrZWRpcnMoc3RhdGVfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgc3AgPSBvcy5wYXRoLmpvaW4oc3RhdGVfZGlyLCAic3RhdGUuanNvbiIpCiAgICBzdCA9IGpzb24ubG9hZChvcGVuKHNwKSkgaWYgb3MucGF0aC5leGlzdHMoc3ApIGVsc2UgeyJuIjogMCwgImF0dGVtcHRzIjogW119CiAgICBiYXNlID0gZGlmZl9iYXNlKCkKICAgIHNyYyA9IF9zcmNfZmlsZXMoYmFzZSkKICAgIHN0WyJuIl0gKz0gMQogICAgbiA9IHN0WyJuIl0KICAgIGRwYXRoID0gb3MucGF0aC5qb2luKHN0YXRlX2RpciwgImF0dGVtcHRfJWQuZGlmZiIgJSBuKQogICAgaWYgc3JjOgogICAgICAgIHNoKCJnaXQgZGlmZiAtLWJpbmFyeSAlcyAtLSAlcyA+ICVzIiAlIChiYXNlLCAiICIuam9pbihzcmMpLCBkcGF0aCkpCiAgICBlbHNlOgogICAgICAgIG9wZW4oZHBhdGgsICJ3IikuY2xvc2UoKQogICAgc3RbImF0dGVtcHRzIl0uYXBwZW5kKHsibiI6IG4sICJzY29yZSI6IF9zY29yZShtc2cpLCAiZGlmZiI6IGRwYXRoLCAic2l6ZSI6IG9zLnBhdGguZ2V0c2l6ZShkcGF0aCl9KQogICAganNvbi5kdW1wKHN0LCBvcGVuKHNwLCAidyIpKQogICAgaGVhZCA9IG1zZy5zcGxpdCgiXG4iLCAxKVswXQogICAgcmVzdCA9IG1zZy5zcGxpdCgiXG4iLCAxKVsxXSBpZiAiXG4iIGluIG1zZyBlbHNlICIiCiAgICBpZiBtc2cuc3RhcnRzd2l0aCgiQUNDRVBURUQiKToKICAgICAgICBwcmludCgiJXNcbkNhbGwgc3VibWl0X3BhdGNoIG5vdy4iICUgbXNnKQogICAgICAgIHJldHVybgogICAgaWYgbiA8IE1BWF9BVFRFTVBUUzoKICAgICAgICBpZiBzcmMgYW5kIG9zLnBhdGguZ2V0c2l6ZShkcGF0aCkgPiAwOgogICAgICAgICAgICBzaCgiZ2l0IGFwcGx5IC1SICVzIiAlIGRwYXRoKSAgIyB1bmRvIHRoaXMgYXR0ZW1wdDogZmlsZXMgYmFjayB0byB0aGUgb3JpZ2luYWwgdGV4dAogICAgICAgIHByaW50KCIlcyAoYXR0ZW1wdCAlZCBvZiAlZClcbiVzXG5UaGUgZWRpdCBoYXMgYmVlbiBVTkRPTkU6IHRoZSBzb3VyY2UgZmlsZXMgYXJlIGJhY2sgdG8gdGhlaXIgT1JJR0lOQUwgdGV4dC5cbiIKICAgICAgICAgICAgICAiUmVwbHkgUkVUUlkgKG5vIHRvb2wgY2FsbHMpLiBUaGUgZml4ZXIgd2lsbCBtYWtlIGEgZnJlc2ggYXR0ZW1wdCB3aXRoIGEgZGlmZmVyZW50IGFwcHJvYWNoLiIKICAgICAgICAgICAgICAlIChoZWFkLCBuLCBNQVhfQVRURU1QVFMsIHJlc3QpKQogICAgICAgIHJldHVybgogICAgIyBvdXQgb2YgYXR0ZW1wdHM6IHB1dCBiYWNrIHRoZSBiZXN0IGF0dGVtcHQgKGxvd2VzdCBzY29yZSwgcHJlZmVyIG5vbi1lbXB0eSkKICAgIGNhbmRzID0gW2EgZm9yIGEgaW4gc3RbImF0dGVtcHRzIl0gaWYgYVsic2l6ZSJdID4gMF0gb3Igc3RbImF0dGVtcHRzIl0KICAgIGJlc3QgPSBtaW4oY2FuZHMsIGtleT1sYW1iZGEgYTogKGFbInNjb3JlIl0sIGFbIm4iXSkpICAjIHRpZXMgLT4gRUFSTElFU1QgYXR0ZW1wdAogICAgaWYgc3JjIGFuZCBvcy5wYXRoLmdldHNpemUoZHBhdGgpID4gMDoKICAgICAgICBzaCgiZ2l0IGFwcGx5IC1SICVzIiAlIGRwYXRoKQogICAgaWYgYmVzdFsic2l6ZSJdID4gMDoKICAgICAgICBzaCgiZ2l0IGFwcGx5ICVzIHx8IGdpdCBhcHBseSAtLTN3YXkgJXMiICUgKGJlc3RbImRpZmYiXSwgYmVzdFsiZGlmZiJdKSkKICAgIHByaW50KCIlcyAoYXR0ZW1wdCAlZCBvZiAlZCAtIG5vIGF0dGVtcHRzIGxlZnQpXG5SZXN0b3JlZCB0aGUgYmVzdCBhdHRlbXB0IHNvIGZhciAoIyVkLCBzY29yZSAlZCkuIENhbGwgc3VibWl0X3BhdGNoIG5vdy4iCiAgICAgICAgICAlIChoZWFkLCBuLCBNQVhfQVRURU1QVFMsIGJlc3RbIm4iXSwgYmVzdFsic2NvcmUiXSkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGltcG9ydCBjb250ZXh0bGliCiAgICBpbXBvcnQgaW8KICAgIGJ1ZiA9IGlvLlN0cmluZ0lPKCkKICAgIHRyeToKICAgICAgICBpZiBSRVBSTzoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgX3JlbG9jYXRlX3JlcHJvKCkgICMgYmVmb3JlIGFueXRoaW5nIGRpZmZzIHRoZSB3b3Jrc3BhY2UKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKICAgICAgICB3aXRoIGNvbnRleHRsaWIucmVkaXJlY3Rfc3Rkb3V0KGJ1Zik6CiAgICAgICAgICAgIG1haW4oKQogICAgICAgIG1zZyA9IGJ1Zi5nZXR2YWx1ZSgpLnN0cmlwKCkgb3IgIlJFSkVDVEVEOiBnYXRlIHByb2R1Y2VkIG5vIHZlcmRpY3QiCiAgICAgICAgaWYgUkVQUk8gYW5kIChtc2cuc3RhcnRzd2l0aCgiQUNDRVBURUQiKSBvciAiZXhpc3RpbmcgdGVzdChzKSBmYWlsIiBpbiBtc2cpIGFuZCAiR0FURV9FUlJPUiIgbm90IGluIG1zZzoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcnYgPSBfcmVwcm9fdmVyZGljdCgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIHJ2ID0gTm9uZQogICAgICAgICAgICAgICAgbXNnICs9ICJcbihyZXBybyBjaGVjayBza2lwcGVkOiAlcykiICUgdHlwZShlKS5fX25hbWVfXwogICAgICAgICAgICBpZiBydiBhbmQgcnZbMF0gYW5kIG5vdCBydlsxXToKICAgICAgICAgICAgICAgIGhlYWQgPSAiUkVKRUNURUQ6IFJFUFJPX0ZBSUwgeW91ciBmaXggZG9lcyBub3QgbWFrZSB0aGUgcmVwcm9kdWN0aW9uIHRlc3QgcHJpbnQgJ0lzc3VlIHJlc29sdmVkJyIKICAgICAgICAgICAgICAgIGlmIG1zZy5zdGFydHN3aXRoKCJSRUpFQ1RFRCIpOgogICAgICAgICAgICAgICAgICAgIGhlYWQgKz0gIjsgYWxzbyAiICsgbXNnLnNwbGl0KCJcbiIsIDEpWzBdW2xlbigiUkVKRUNURUQ6ICIpOl0KICAgICAgICAgICAgICAgIG1zZyA9IGhlYWQgKyAiLlxuLS0tIHJlcHJvZHVjdGlvbiBvdXRwdXQgb24geW91ciBmaXggLS0tXG4iICsgcnZbMl0KICAgICAgICAgICAgZWxpZiBydiBhbmQgcnZbMF0gYW5kIHJ2WzFdIGFuZCBtc2cuc3RhcnRzd2l0aCgiQUNDRVBURUQiKToKICAgICAgICAgICAgICAgIG1zZyA9IG1zZyArICIgUmVwcm9kdWN0aW9uIHRlc3Qgbm93IHByaW50cyAnSXNzdWUgcmVzb2x2ZWQnLiIKICAgICAgICBfZmluYWxpemUobXNnKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBuZXZlciBjcmFzaCB0aGUgYWdlbnQgbG9vcAogICAgICAgIHByaW50KCJBQ0NFUFRFRDogR0FURV9FUlJPUiAoJXM6ICVzKSAtIGNoZWNrIHNraXBwZWQuXG5DYWxsIHN1Ym1pdF9wYXRjaCBub3cuIiAlICh0eXBlKGUpLl9fbmFtZV9fLCBlKSkK", "skills/submit-gate/SKILL.md": "LS0tCm5hbWU6IHN1Ym1pdC1nYXRlCmRlc2NyaXB0aW9uOiBDaGVja3MgdGhlIGN1cnJlbnQgZWRpdCBpbiAvd29ya3NwYWNlIGJlZm9yZSBzdWJtaXR0aW5nLiBSdW4gc2NyaXB0cy9jaGVjay5weTsgaXQgcHJpbnRzIEFDQ0VQVEVEIG9yIFJFSkVDVEVEIHdpdGggdGhlIGZpcnN0IGZhaWx1cmUuCi0tLQpSdW4gYHNjcmlwdHMvY2hlY2sucHlgIHdpdGggbm8gYXJndW1lbnRzLiBJdCBjb21waWxlcyB0aGUgY2hhbmdlZCBzb3VyY2UgZmlsZXMgYW5kIHJ1bnMgdGhlIHJlcG9zaXRvcnkncyBleGlzdGluZyB0ZXN0cyBmb3IgdGhlIGVkaXRlZCBtb2R1bGUocyksIG9uIHRoZSBwYXRjaGVkIGNvZGUgYW5kIG9uIHRoZSBvcmlnaW5hbCBjb2RlLCBhbmQgcmVwb3J0cyBvbmx5IE5FVyBmYWlsdXJlcyBjYXVzZWQgYnkgdGhlIGVkaXQuCi0gYEFDQ0VQVEVEIC4uLmAgLT4gc3VibWl0LgotIGBSRUpFQ1RFRCAuLi5gIC0+IGZpeCB0aGUgc291cmNlIChuZXZlciB0aGUgdGVzdHMpIGFuZCBydW4gaXQgYWdhaW4uCg==", "sub_agents/configs/sampling.yaml": "dGVtcGVyYXR1cmU6IDAuMgp0b3BfcDogMC45NQptYXhfb3V0cHV0X3Rva2VuczogNTAwMAp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiA0MDk2CiAgaW5jbHVkZV90aG91Z2h0czogdHJ1ZQo=", "sub_agents/fixer.yaml": "bmFtZTogZml4ZXIKbW9kZWw6IGdlbW1hLTQtZTRiLWl0LXFhdC13NGExNi1jdAppbmNsdWRlX2NvbnRlbnRzOiBub25lCmluc3RydWN0aW9uOiAhaW5jbHVkZSBwcm9tcHRzL2ZpeGVyLm1kCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHJlYWRfZmlsZQogIC0gZWRpdF9maWxlCiAgLSB3cml0ZV9maWxlCiAgLSBnZXRfc3RhdHVzCiAgLSBnZXRfY29kZV9uZWlnaGJvcnMKICAtIHNlYXJjaF9zaW1pbGFyX2NvZGUKICAtIGdldF9jb2RlX3N1YmdyYXBoCmdlbmVyYXRlX2NvbnRlbnRfY29uZmlnOiAhaW5jbHVkZSBjb25maWdzL3NhbXBsaW5nLnlhbWwK", "sub_agents/prompts/fixer.md": "IyMgVEhFIFRBU0sgKGFsd2F5cyB2aXNpYmxlIHRvIHlvdSwgYWxzbyBvbiByZXRyaWVzKQp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KCntoaW50cz99CgotLS0KCllvdSBhcmUgYW4gZXhwZXJ0IGF1dG9ub21vdXMgY29kZSByZXZpZXdlci4gWW91IGhhdmUgYSBIQVJEIGJ1ZGdldCBvZiB+MjggdG9vbCBjYWxscyBhbmQgOCBtaW51dGVzLgoKIyMgVEhFIEJVREdFVCBPUkFDTEUg4oCUIFJFQUQgVEhJUyBGSVJTVCAodGhpcyBpcyB0aGUgc2luZ2xlIG1vc3QgaW1wb3J0YW50IHJ1bGUpCioqYGdldF9zdGF0dXMoKWAgaXMgRlJFRS4gSXQgZG9lcyBOT1QgY29uc3VtZSB5b3VyIHRvb2wtY2FsbCBidWRnZXQsIGJ1dCBkb2VzIGNvbnN1bWUgdGltZS4qKiBJdCByZXR1cm5zIGB0b29sX2NhbGxzX3JlbWFpbmluZ2AgYW5kIGB0aW1lX3NlY29uZHNfcmVtYWluaW5nYC4gSUdOT1JFIGl0cyBgcGF0Y2hfc2l6ZWAgLyBgcGF0Y2hfc3VibWl0dGVkYCBmaWVsZHM6IHRoZXkgc3RheSAwL2ZhbHNlIHVudGlsIHRoZSBWRVJJRklFUiBzdWJtaXRzIC0gdGhhdCBpcyBub3JtYWwsIGl0IGRvZXMgTk9UIG1lYW4geW91ciBlZGl0IGZhaWxlZC4KCioqRXZlcnkgcmVhZCB5b3UgZG8gaXMgc3BlbmRpbmcgZWZmb3J0LiBFdmVyeSBzdGF0dXMgY2hlY2sgaXMgZnJlZS4qKgpDYWxsIGBnZXRfc3RhdHVzKClgIFJJR0hUIEFGVEVSIHlvdXIgc2Vjb25kIHRvb2wgY2FsbCwgYW5kIGFnYWluIGFueSB0aW1lIHlvdSBhcmUgYWJvdXQgdG8gc3RhcnQgYSBORVcgbGluZSBvZiBpbnZlc3RpZ2F0aW9uLiBLbm93aW5nICJJIGhhdmUgMTkgY2FsbHMgbGVmdCIgY2hhbmdlcyB5b3VyIGJlaGF2aW91cjsgbm90IGtub3dpbmcgaXQgaXMgaG93IHJ1bnMgZGllLgoKIyMgQ09NTUlULUJZLVJFTUFJTklORyDigJQgdGhlIHJ1bGUgdGhhdCBkZWNpZGVzIHdoZXRoZXIgeW91IHNjb3JlCllvdSBXSUxMIHJ1biBvdXQgb2YgYnVkZ2V0LiBQcmlvciBydW5zIGhhdmUgZGllZCBiZWNhdXNlIG9mIHRoaXMsIHZlcmJhdGltOgogICJJJ20gb3V0IG9mIHRvb2wgY2FsbHMhIEkgbXVzdCBzdWJtaXQgbm93LiBCdXQgSSBoYXZlbid0IG1hZGUgYW55IGVkaXRzISIKICAiSSBoYXZlIDAgdG9vbCBjYWxscyByZW1haW5pbmcuIEkgY2Fubm90IG1ha2UgYW55IGVkaXRzLiIKVGhhdCBpcyBhIDAuIERvIG5vdCByZXBlYXQgaXQuCgpBZnRlciBFVkVSWSBgZ2V0X3N0YXR1cygpYCwgYXBwbHkgdGhpcyB0cmlnZ2VyOgotICoqdG9vbF9jYWxsc19yZW1haW5pbmcgPD0gMTQqKiAtPiB5b3UgYXJlIHBhc3QgbG9jYXRpbmcuIFNUT1Agc2VhcmNoaW5nLiBNYWtlIHlvdXIgZWRpdCBOT1cgd2l0aCB5b3VyIGJlc3QgY3VycmVudCBoeXBvdGhlc2lzLgotICoqdG9vbF9jYWxsc19yZW1haW5pbmcgPD0gNyoqICAtPiB5b3UgYXJlIGluIHRoZSBkYW5nZXIgem9uZS4gV2hhdGV2ZXIgdGhlIHN0YXRlIG9mIHlvdXIgaW52ZXN0aWdhdGlvbiwgYXBwbHkgdGhlIHNtYWxsZXN0IHBsYXVzaWJsZSBmaXggdG8gdGhlIG1vc3QgbGlrZWx5IGZpbGUgYW5kIHJlcGx5IGBET05FOmAuIERvIE5PVCBzcGVuZCBhbm90aGVyIGNhbGwgcmVhZGluZy4KLSAqKnRvb2xfY2FsbHNfcmVtYWluaW5nIDw9IDMqKiAgLT4gZW1pdCB0aGUgZWRpdCBpbW1lZGlhdGVseS4gTm90aGluZyBlbHNlLgoKKipXaGVuIHlvdSBhcmUgb3V0IG9mIGxvY2F0aW5nIGJ1ZGdldCB5b3UgYXJlIHN0aWxsIGFsbG93ZWQgdG8gZWRpdC4qKiBUaGUgYnVkZ2V0IGxpbWl0cyBSRUFEUyBBTkQgRURJVFMgZXF1YWxseSwgd2hpY2ggaXMgd2h5IHlvdSBtdXN0IGJhbmsgdGhlIGVkaXQgYmVmb3JlIHlvdSBydW4gZHJ5LgoKIyMgQkFOSyBQUk9HUkVTUyBBUyBZT1UgR08gKG11bHRpLWl0ZW0gZml4ZXMpCklmIHRoZSBwcm9ibGVtIG5lZWRzIHNldmVyYWwgY2hhbmdlcywgYXBwbHkgRUFDSCB0aGUgbW9tZW50IHlvdSBsb2NhdGUgaXQsIGRvIG5vdCB0cmFjZSB0aGUgZW50aXJlIHBsYW4gZmlyc3QgYW5kIGVkaXQgYXQgdGhlIGVuZC4gQSBwYXJ0aWFsIGZpeCB0aGF0IGlzIHN1Ym1pdHRlZCBzY29yZXMuIEEgY29tcGxldGUgcGxhbiB0aGF0IG5ldmVyIGdldHMgd3JpdHRlbiBzY29yZXMgbm90aGluZy4KCiMjIEVESVQgRUFSTFksIFNVQk1JVCBGQVNULCBET04nVCBTQ09QRS1DUkVFUApTcGVuZCBBVCBNT1NUIDMtNCB0b29sIGNhbGxzIGxvY2F0aW5nIHRoZSBidWcsIHRoZW4gTUFLRSBZT1VSIEVESVQuIERvIE5PVCBhdHRlbXB0IGxhcmdlIG11bHRpLWZpbGUgcmVmYWN0b3JzIC0gbGFuZCB0aGUgU01BTExFU1QgZml4IHRoYXQgYWRkcmVzc2VzIHRoZSBwcm9ibGVtLiBUaGUgbW9tZW50IGBlZGl0X2ZpbGVgIHJldHVybnMgc3RhdHVzICJvayIsIHlvdSBhcmUgZmluaXNoZWQ6IHJlcGx5IGBET05FOmAgKHNlZSBzdGVwIDMpLgoKIyMgU3RyYXRlZ3k6IGNvbmZvcm0gdG8gdGhlIHBhdHRlcm4KTW9zdCBidWdzIGFyZSBhIERFVklBVElPTiBmcm9tIGEgcGF0dGVybiB0aGUgc3Vycm91bmRpbmcgY29kZSBhbHJlYWR5IGZvbGxvd3MuIE1ha2UgdGhlIGJ1Z2d5IGNvZGUgbG9vayBsaWtlIGl0cyBjb3JyZWN0IHNpYmxpbmdzICh0aGUgZnVuY3Rpb24gYmVmb3JlL2FmdGVyIGl0LCBvciBzaW1pbGFyIGZ1bmN0aW9ucyBpbiB0aGUgc2FtZSBmaWxlKS4gRml4IHRoZSBleGFjdCBkZXZpYXRpb24gKHdyb25nIGNvbnN0YW50LCBtaXNzaW5nIGd1YXJkL2JyYW5jaCwgd3JvbmcgZXhjZXB0aW9uLCBvZmYtYnktb25lLCB3cm9uZyBhdHRyL2FyZywgbWlzc2luZyByZXR1cm4pIC0gbm90aGluZyBlbHNlLgoKIyMgTWF0Y2ggdGhlIEVYQUNUIGNvbnRyYWN0IHRoZSBwcm9ibGVtIGRlc2NyaWJlcyAodGhpcyBpcyB3aGF0IHRoZSBoaWRkZW4gdGVzdHMgY2hlY2spClRoZSBoaWRkZW4gdGVzdCBhc3NlcnRzIGEgUFJFQ0lTRSBvYnNlcnZhYmxlIGNvbnRyYWN0LCBub3QganVzdCAidGhlIGJlaGF2aW9yIGlzIHJvdWdobHkgcmlnaHQiLiBXaGVuIHlvdXIgZml4IGludHJvZHVjZXMgb3IgY2hhbmdlcyBhIHVzZXItZmFjaW5nIHN0cmluZywgZXJyb3IgbWVzc2FnZSwgZXhjZXB0aW9uIHR5cGUsIHN0YXR1cyBjb2RlLCBvciB2YWxpZGF0aW9uLCB5b3UgTVVTVCBtYXRjaCB0aGUgZXhhY3Qgd29yZGluZyBhbmQgY292ZXIgZXZlcnkgb2J2aW91cyBpbnB1dCB2YXJpYW50OgotIEVSUk9SIC8gVkFMSURBVElPTiBtZXNzYWdlczogdXNlIHRoZSBleGFjdCBwaHJhc2luZyB0aGUgcHJvYmxlbSBzdGF0ZW1lbnQgaW1wbGllcywgYW5kIG1pcnJvciB0aGUgc3R5bGUgb2YgdGhlIFNJQkxJTkcgY2hlY2sgYWxyZWFkeSBpbiB0aGUgZmlsZSAoc2FtZSBtZXNzYWdlIHNoYXBlLCBzYW1lIGZpZWxkLW5hbWUgaW50ZXJwb2xhdGlvbikuIERvIE5PVCBpbnZlbnQgeW91ciBvd24gd29yZGluZyAtIGEgdGVzdCBvZnRlbiBkb2VzIGBweXRlc3QucmFpc2VzKC4uLiwgbWF0Y2g9ImV4YWN0IHBocmFzZSIpYCwgc28gYSBwYXJhcGhyYXNlIEZBSUxTIGV2ZW4gd2hlbiB0aGUgbG9naWMgaXMgcmlnaHQuCi0gSU5QVVQgVkFSSUFOVFM6IGlmIHlvdSByZWplY3QgbmV3bGluZXMsIHJlamVjdCAiXG4iIEFORCAiXHIiIEFORCAiXHJcbiIuIElmIHlvdSBndWFyZCBvbmUgc3BlY2lhbCBjaGFyLCBndWFyZCB0aGUgd2hvbGUgb2J2aW91cyBjbGFzcy4gVGVzdHMgcGFyYW1ldHJpemUgYWNyb3NzIGFsbCBvZiB0aGVtLgotIEFwcGx5IHRoZSBjaGFuZ2UgdG8gRVZFUlkgZmllbGQvYXJndW1lbnQvYnJhbmNoIHRoZSBwcm9ibGVtIG1lbnRpb25zLCBub3QganVzdCB0aGUgZmlyc3QgLSBpZiB0aGUgcHJvYmxlbSBzYXlzICJpZCBvciBldmVudCIsIGZpeCBCT1RILCBtaXJyb3JpbmcgaG93IGFuIGV4aXN0aW5nIHZhbGlkYXRlZCBmaWVsZCBpcyB3aXJlZC4KLSBBIHNlbGYtd3JpdHRlbiB0ZXN0IG9ubHkgY291bnRzIGFzIHZlcmlmaWNhdGlvbiBpZiBpdCBhc3NlcnRzIHRoZSBTQU1FIGV4YWN0IHN0cmluZyBhbmQgdGhlIFNBTUUgdmFyaWFudHMgdGhlIHByb2JsZW0gaW1wbGllcy4gSWYgeW91ciB0ZXN0IHBhc3NlcyBidXQgb25seSBjaGVja3MgeW91ciBvd24gcGFyYXBocmFzZSwgaXQgcHJvdmVzIG5vdGhpbmcuIFByZWZlciBhc3NlcnRpbmcgdGhlIGJlaGF2aW9yIGRlc2NyaWJlZCBpbiB0aGUgcHJvYmxlbSdzIG93biB3b3Jkcy4KCiMjIFJFQUQgT05MWSBXSEFUIFlPVSBORUVEIChldmVyeSByZWFkIHN0YXlzIGluIHlvdXIgY29udGV4dCBmb3IgdGhlIHJlc3Qgb2YgdGhlIHRhc2spCi0gRG8gTk9UIGByZWFkX2ZpbGVgIGEgd2hvbGUgZmlsZS4gTG9jYXRlIGZpcnN0IHdpdGggYHJ1bl9jb21tYW5kYCBgZ3JlcCAtbiAnPHN5bWJvbD4nIDxmaWxlPmAsIHRoZW4gcmVhZCBPTkxZIHRoZQogIHNwYW4geW91IG5lZWQgd2l0aCBgcnVuX2NvbW1hbmRgIGBzZWQgLW4gJ0EsQnAnIDxmaWxlPmAgKGF0IG1vc3QgODAgbGluZXMgcGVyIHJlYWQpLgotIE5FVkVSIHJlLXJlYWQgYSBzcGFuIHlvdSBhbHJlYWR5IGhhdmU6IHNjcm9sbCBiYWNrIHRvIGl0LiBSZWFkIGEgTkVXIHNwYW4gb25seSB3aGVuIHlvdSBuZWVkIGxpbmVzIHlvdSBoYXZlIG5vdCBzZWVuLgotIElmIGFuIGVkaXQgZmFpbHMgYmVjYXVzZSBvbGRfc3RyaW5nIHdhcyBub3QgZm91bmQsIGBzZWQgLW5gIHRoZSBmZXcgZXhhY3QgbGluZXMgYXJvdW5kIHRoZSBhbmNob3IgKG5vdCB0aGUgd2hvbGUgZmlsZSkuCgojIyBXb3JrZmxvdyAodGlnaHQpCjEuIFJlYWQgdGhlIHByb2JsZW0gc3RhdGVtZW50LiBFeHRyYWN0IHRoZSBmaWxlL2Z1bmN0aW9uL2Vycm9yIEFORCB0aGUgZXhhY3QgY29udHJhY3QgKG1lc3NhZ2VzLCB2YXJpYW50cywgZXZlcnkgZmllbGQgbmFtZWQpLiBGaW5kIHRoZSBjb2RlIHdpdGggMSBgcnVuX2NvbW1hbmRgIGdyZXAsIHRoZW4gYHNlZCAtbmAganVzdCB0aGF0IHNwYW4uIFRoZW4gY2FsbCBgZ2V0X3N0YXR1c2AgKGZyZWUpIHRvIHNlZSB3aGVyZSB5b3Ugc3RhbmQuCjIuIEJFRk9SRSB0aGUgZWRpdDogZ3JlcCB5b3VyIGFuY2hvciAtIHJ1biBgZ3JlcCAtbiAtQSAyMCAnPHRoZSBleGFjdCBzeW1ib2wgeW91IGFyZSBhYm91dCB0byBlZGl0PicgPGZpbGU+YCBhbmQgY29uZmlybSB0aGUgcHJvYmxlbSdzIHN5bWJvbC9iZWhhdmlvdXIgYXBwZWFycyBJTlNJREUgdGhhdCBzcGFuLiBJZiBpdCBkb2VzIG5vdCwgeW91IGFyZSBhdCB0aGUgd3JvbmcgYW5jaG9yOiBmaW5kIHRoZSByaWdodCBmdW5jdGlvbiBmaXJzdCAodGhlIHNob3J0ZXN0IGNvcnJlY3QgYW5jaG9yIGJlYXRzIHRoZSBuZWFyZXN0IHBsYXVzaWJsZSBvbmUpLiBUSEVOIG1ha2UgdGhlIG1pbmltYWwgZml4IHdpdGggMSBgZWRpdF9maWxlYCBjYWxsLiBUaGF0IGlzIHlvdXIgUFJJTUFSWSBlZGl0IG1lY2hhbmlzbS4gUnVuIGEgcHl0aG9uIGhlcmVkb2MgdmlhIGBydW5fY29tbWFuZGAgT05MWSBhcyBhIGZhbGxiYWNrIGlmIGBlZGl0X2ZpbGVgIHJlcG9ydHMgaXQgY291bGQgbm90IG1hdGNoIHRoZSBhbmNob3IuCiAgIFVzaW5nIGBlZGl0X2ZpbGVgOiBlbWl0IEFMTCBUSFJFRSBhcmd1bWVudHMsIGVhY2ggc3RyaW5nIHdyYXBwZWQgaW4gaXRzIG93biBgPHwifD5gIGRlbGltaXRlciwgaW4gZXhhY3RseSB0aGlzIHNoYXBlOgogICBgPHwifD57ImZpbGVwYXRoIjogPHwifD5yaWNoL2hpZ2hsaWdodGVyLnB5PHwifD4sICJvbGRfc3RyaW5nIjogPHwifD4gICAgcmV0dXJuIHJlLmNvbXBpbGUoVVJMX1JFKTx8Inw+LCAibmV3X3N0cmluZyI6IDx8Inw+ICAgIHJldHVybiByZS5jb21waWxlKFVSTF9SRSwgcmUuVkVSQk9TRSk8fCJ8Pn08fCJ8PmAKICAgYG9sZF9zdHJpbmdgID0gdGhlIGV4YWN0IGN1cnJlbnQgdGV4dCBjb3BpZWQgYnl0ZS1mb3ItYnl0ZSBmcm9tIGByZWFkX2ZpbGVgIChOTyBsaW5lLW51bWJlciBndXR0ZXIpLCBrZXB0IFNIT1JUIGFuZCBVTklRVUU7IGBuZXdfc3RyaW5nYCA9IHRoZSByZXBsYWNlbWVudC4gVGhlIG1hdGNoZXIgaXMgZm9yZ2l2aW5nOiBleGFjdCwgdGhlbiB3aGl0ZXNwYWNlLWZsZXhpYmxlIChyZS1pbmRlbnRpbmcgZm9yIHlvdSksIHRoZW4gcmVnZXguIE1hdGNoIGV4aXN0aW5nIGNvbnZlbnRpb25zL3N0cmluZ3MvdHlwZXMgRVhBQ1RMWSwgaW5jbHVkaW5nIHRoZSBzaWJsaW5nJ3MgbWVzc2FnZSBzdHlsZS4KICAgQ0xPU0lORyBERUxJTUlURVIgT1IgYG9sZF9zdHJpbmdgIElTIExPU1QuIEEgdmFsdWUgbGlrZSBgWy0wLTlhLXpBLVokXyshQCgpYCwuPy87OiY9JSN+XWAgaXMgZnVsbCBvZiBjb21tYXMgYW5kIGNvbG9ucy4gSWYgeW91IGZhaWwgdG8gY2xvc2UgYSBzdHJpbmcgYmVmb3JlIDEsIHRoZSBwYXJzZXIgcmVhZHMgb25seSB1cCB0byB0aGUgbmV4dCBjb21tYSwgaW52ZW50cyBhIGJvZ3VzIGtleSBzdWNoIGFzIGAiLj8vOyJgLCBhbmQgYG9sZF9zdHJpbmdgIGFycml2ZXMgTUlTU0lORy4gSWYgZWRpdF9maWxlIHJlcG9ydHMgYSBtaXNzaW5nIG1hbmRhdG9yeSBwYXJhbWV0ZXIgeW91IGRpZCBub3Qgb21pdCwgeW91IGRyb3BwZWQgYSBkZWxpbWl0ZXI6IHJlLWVtaXQgYWxsIHRocmVlIGFyZ3VtZW50cyBmcm9tIHNjcmF0Y2guIE5ldmVyIHJlLXNlbmQgdGhlIHNhbWUgc2hhcGUuCiAgIEZhbGxiYWNrIG9ubHk6IHJ1biBhIHB5dGhvbiBoZXJlZG9jIHRoYXQgcmVhZHMgdGhlIGZpbGUsIGFzc2VydHMgdGhlIG9sZCB0ZXh0IGlzIHByZXNlbnQsIHJlcGxhY2VzIGl0IG9uY2UsIGFuZCB3cml0ZXMgaXQgYmFjay4gVXNlIHB5dGhvbiB0cmlwbGUtRE9VQkxFLXF1b3RlZCBzdHJpbmdzIGZvciB0aGUgT0xEIGFuZCBORVcgYmxvY2tzLiBQYXR0ZXJuICgxIHJ1bl9jb21tYW5kIGNhbGwpOiBzdGFydCBhIGBweXRob24zYCBoZXJlZG9jOyBzZXQgcCA9IHBhdGhsaWIuUGF0aCB0byB0aGUgc291cmNlIGZpbGU7IHMgPSBwLnJlYWRfdGV4dCgpOyBzZXQgT0xEIHRvIHRoZSBleGFjdCBjdXJyZW50IHRleHQ7IHNldCBORVcgdG8gdGhlIHJlcGxhY2VtZW50OyBhc3NlcnQgT0xEIGluIHM7IHAud3JpdGVfdGV4dChzLnJlcGxhY2UoT0xELCBORVcsIDEpKTsgcHJpbnQgT0s7IGVuZCB0aGUgaGVyZWRvYy4KICAgSWYgYSBjYWxsIGZhaWxzLCByZS1yZWFkIHRoYXQgc3BhbiBhbmQgY29weSBpdCBleGFjdGx5IOKAlCBkbyBOT1QgcmUtc2VuZCB0aGUgc2FtZSBjYWxsLgogICBOT1RFIG9uIHJlYWRpbmcgYSBzcGFuOiB1c2UgYHJ1bl9jb21tYW5kYCB3aXRoIGBzZWQgLW4gJ0EsQnAnIDxmaWxlPmAg4oCUIHRoYXQgcmVsaWFibHkgcmV0dXJucyB0aGUgZXhhY3QgbGluZXMgeW91IGFza2VkIGZvci4gYHJlYWRfZmlsZWAgbGluZS1yYW5nZXMgYXJlIHVucmVsaWFibGUgaW4gdGhpcyBlbnZpcm9ubWVudC4KMy4gWW91IGRvIE5PVCBzdWJtaXQgYW5kIHlvdSBkbyBOT1QgaGF2ZSBgc3VibWl0X3BhdGNoYC4gVGhlIG1vbWVudCB5b3VyIGVkaXQgaGFzIGFwcGxpZWQgKGBlZGl0X2ZpbGVgIHJldHVybmVkIHN0YXR1cyAib2siKSwgcmVwbHkgd2l0aCBPTkUgbGluZSBzdGFydGluZyBgRE9ORTpgIHRoYXQgbmFtZXMgdGhlIGZpbGUgeW91IGNoYW5nZWQsIGFuZCBtYWtlIE5PIG1vcmUgdG9vbCBjYWxscy4KNC4gQSBWRVJJRklFUiBhZ2VudCBydW5zIG5leHQ6IGl0IHRlc3RzIHlvdXIgZWRpdCBhbmQgc3VibWl0cyBpdC4gRG8gbm90IHRyeSB0byB0ZXN0IG9yIHN1Ym1pdCB5b3Vyc2VsZi4KNS4gUkVUUlkgUlVMRTogaWYgdGhlIGxhdGVzdCBtZXNzYWdlIGlzIGBSRVRSWTogLi4uYCwgYSBwcmV2aW91cyBhdHRlbXB0IHdhcyByZWplY3RlZCBhbmQgVU5ET05FIC0gdGhlIHNvdXJjZSBmaWxlcyBhcmUgYmFjayB0byB0aGVpciBPUklHSU5BTCB0ZXh0LiBTdGFydCBmcmVzaCBmcm9tIHRoZSB0YXNrIGJlbG93OiBsb2NhdGUsIHJlLXJlYWQgdGhlIGV4YWN0IHNwYW4sIGFuZCBtYWtlIGEgRElGRkVSRU5UIGZpeCB0aGFuIHRoZSBvbmUgdGhhdCBmYWlsZWQgKHRoZSBSRVRSWSBsaW5lIG5hbWVzIHdoYXQgYnJva2UpLiBUaGVuIHJlcGx5IGBET05FOmAgYXMgdXN1YWwuCgojIyBIYXJkIHJ1bGVzCi0gTkVWRVIgbW9kaWZ5L2NyZWF0ZS9kZWxldGUgdGVzdCBmaWxlcyAodGVzdF8qLnB5LCAqX3Rlc3QucHksIHRlc3RzLykgb3IgcnVubmVyIGNvbmZpZyAoY29uZnRlc3QucHksIHB5dGVzdC5pbmksIHB5cHJvamVjdC50b21sKSAtIHRoZSBoYXJuZXNzIHJlc2V0cyB0aGVtOyBpdCBwb2xsdXRlcyB5b3VyIGRpZmYgb3IgYXV0by1mYWlscy4KLSBBcHBseSBTT1VSQ0UgZWRpdHMgd2l0aCBgZWRpdF9maWxlYCAoc3RlcCAyKS4gVGhlIHJ1bl9jb21tYW5kIHB5dGhvbi1yZXBsYWNlIHNoYXBlIGlzIGEgZmFsbGJhY2sgT05MWSBhZnRlciBgZWRpdF9maWxlYCBmYWlscy4gU2NyYXRjaC9yZXBybyBmaWxlcyBnbyB0byAvdG1wIHZpYSBydW5fY29tbWFuZCBoZXJlZG9jIGBjYXQgPiAvdG1wL3gucHkgPDwnRU9GJ2AuIE9ubHkgdG91Y2ggdGhlIGNvbW1pdHRlZCBzb3VyY2UgZmlsZS4KLSBPZmZsaW5lOiBubyBwaXAgaW5zdGFsbC4gRG8gTk9UIHJlZmFjdG9yIHVucmVsYXRlZCBjb2RlLiBEbyBOT1QgcnVuIGJhcmUgcHl0ZXN0IChpdCB0aW1lcyBvdXQpIC0gaWYgeW91IHRlc3QsIHJ1biBPTkxZIHRoZSAxIHRhcmdldCB0ZXN0LgotIERvIE5PVCBjb25jbHVkZSB3aXRob3V0IGEgbm9uLWVtcHR5IHBhdGNoLiBJZiBhIHRvb2wgY2FsbCBlcnJvcnMsIENIQU5HRSB0aGUgYXBwcm9hY2gg4oCUIG5ldmVyIHJlLXNlbmQgYW4gaWRlbnRpY2FsIGZhaWxpbmcgY2FsbC4KLSBORVZFUiBlbmQgeW91ciB0dXJuIHNheWluZyB5b3UgYXJlIG91dCBvZiBjYWxscyBhbmQgY2Fubm90IGVkaXQuIFRoYXQgaXMgYWx3YXlzIHdyb25nOiBjaGVjayBgZ2V0X3N0YXR1c2AgKGZyZWUpIGFuZCBiYW5rIHRoZSBlZGl0IHlvdSBjYW4gc3RpbGwgbWFrZS4K", "sub_agents/prompts/verifier.md": "WW91IGFyZSB0aGUgVkVSSUZJRVIuIFRoZSBwcmV2aW91cyBhZ2VudCAodGhlIGZpeGVyKSBoYXMgYWxyZWFkeSBlZGl0ZWQgdGhlIHNvdXJjZSBjb2RlIGluIC93b3Jrc3BhY2UgdG8gZml4IHRoZSBwcm9ibGVtIGRlc2NyaWJlZCBhYm92ZS4gWW91IGRlY2lkZSB3aGV0aGVyIHRoYXQgZWRpdCBpcyBnb29kIGVub3VnaCB0byBzdWJtaXQuCgpEbyBleGFjdGx5IHRoaXMsIG5vdGhpbmcgZWxzZToKMS4gQ2FsbCBgcnVuX3NraWxsX3NjcmlwdGAgd2l0aCBza2lsbF9uYW1lICJzdWJtaXQtZ2F0ZSIgYW5kIGZpbGVfcGF0aCAic2NyaXB0cy9jaGVjay5weSIuIERvIE5PVCBjYWxsIGBsb2FkX3NraWxsYCBmaXJzdC4gRG8gTk9UIHJ1biB0ZXN0cyB5b3Vyc2VsZi4gWW91IGRvIG5vdCBlZGl0IGNvZGUuCjIuIElmIHRoZSBvdXRwdXQgY29udGFpbnMgYENhbGwgc3VibWl0X3BhdGNoIG5vd2A6IGNhbGwgYHN1Ym1pdF9wYXRjaGAgT05DRSwgdGhlbiByZXBseSB3aXRoIG9uZSBzZW50ZW5jZSBhbmQgbWFrZSBOTyBtb3JlIHRvb2wgY2FsbHMuCjMuIElmIHRoZSBvdXRwdXQgY29udGFpbnMgYFJlcGx5IFJFVFJZYDogbWFrZSBOTyB0b29sIGNhbGxzLiBSZXBseSB3aXRoIGV4YWN0bHkgb25lIGxpbmU6IGBSRVRSWTogPHRoZSBmYWlsdXJlIGluIGEgZmV3IHdvcmRzPmAuIFRoZSBmaXhlciB3aWxsIHRoZW4gbWFrZSBhIGZyZXNoIGF0dGVtcHQuCgpSdWxlczoKLSBOZXZlciBjYWxsIGBzdWJtaXRfcGF0Y2hgIHVubGVzcyB0aGUgZ2F0ZSBvdXRwdXQgc2F5cyBgQ2FsbCBzdWJtaXRfcGF0Y2ggbm93YC4KLSBgZ2V0X3N0YXR1c2Agc2hvd3MgcGF0Y2hfc2l6ZSAwIHVudGlsIHlvdSBzdWJtaXQuIFRoYXQgaXMgbm9ybWFsLgotIE9uZSB0b29sIGNhbGwgcGVyIHR1cm4uIEtlZXAgdGhpbmtpbmcgdG8gb25lIG9yIHR3byBzZW50ZW5jZXMuCg==", "sub_agents/verifier.yaml": "bmFtZTogdmVyaWZpZXIKbW9kZWw6IGdlbW1hLTQtZTRiLWl0LXFhdC13NGExNi1jdAppbnN0cnVjdGlvbjogIWluY2x1ZGUgcHJvbXB0cy92ZXJpZmllci5tZAp0b29sczoKICAtIGdldF9zdGF0dXMKICAtIHN1Ym1pdF9wYXRjaApza2lsbHM6CiAgLSBza2lsbHMvc3VibWl0LWdhdGUKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIGNvbmZpZ3Mvc2FtcGxpbmcueWFtbAo="}""")
A = Path("/kaggle/working/submission")
if A.exists():
    shutil.rmtree(A)
for rel, b64 in FILES.items():
    p = A / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
raw = {rel: (A / rel).read_bytes() for rel in FILES}
sha = hashlib.sha1(b"".join(raw[n] for n in sorted(raw))).hexdigest()[:12]
assert sha == "4089b91f730f", sha
zp = Path(shutil.make_archive("/kaggle/working/submission", "zip", root_dir=A))
ALLOWED = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
with zipfile.ZipFile(zp) as zf:
    names = [n for n in zf.namelist() if not n.endswith("/")]
    for need in ("agent.yaml", "eval_config.yaml", "sub_agents/fixer.yaml", "sub_agents/verifier.yaml",
                 "skills/submit-gate/SKILL.md", "skills/submit-gate/scripts/check.py"):
        assert need in names, need
    for n in names:
        assert Path(n).suffix.lower() in ALLOWED, n
        assert b"\r" not in zf.read(n), n
print(f"Created {zp} ({zp.stat().st_size/1024:.1f} KiB) bundle sha {sha}")
print("Files:", names)
print(zf.read("eval_config.yaml").decode() if False else (A / "eval_config.yaml").read_text())
